# BANKING77 Intent Router — LoRA fine-tune of Qwen2.5-0.5B-Instruct

**The use case.** A bank's support chat has to read an incoming customer message
and send it to the right team. There are 77 possible intents, such as
`card_arrival`, `lost_or_stolen_card` or `wrong_amount_of_cash_received`.
Sending a fraud report to the card-delivery queue costs real money, so the
routing has to be accurate.

**What we do.** We take a small pretrained model, measure how well it does this
job out of the box, fine-tune it on 4,000 labelled examples, and measure it
again. We compare three systems on the same test data:

| | System | Prompt it gets |
|---|---|---|
| A | Base model | Short prompt, no list of valid labels |
| B | Base model | Long prompt listing all 77 labels |
| C | Fine-tuned model | Short prompt — it learned the labels from the data |

We then test whether the fine-tuned model treats differently-worded versions of
the same question differently, which is the bias check.

Every number this notebook prints is saved to `results.json`. Those are the
numbers the report is written from.

**Before you run:** in the Kaggle panel on the right, set
**Accelerator = GPU T4 x2** and **Internet = On**. Then Run All.
Expect about 20–25 minutes.

## Step 0 — Install the libraries

`transformers` loads the model, `peft` provides LoRA, `accelerate` handles the
training loop's device placement.

Kaggle ships an old copy of `torchao` that current `peft` refuses to start
alongside. Nothing here uses `torchao`, so we remove it first.

In [ ]:
%pip uninstall -q -y torchao
%pip install -q -U peft transformers accelerate

## Step 1 — Settings

Everything adjustable lives here. If accuracy comes out low, raise `N_TRAIN` and
`EPOCHS` first. `SMOKE=1` is a tiny practice run that works without a GPU.

We deliberately use only one of Kaggle's two GPUs. Given two, the training loop
splits each batch across both and then gathers the results back onto the first
GPU. Those results include one score per token in a 151,936-word vocabulary, so
the gathered block runs to gigabytes and the first GPU runs out of memory. One
GPU is enough for a model this size, and it avoids the whole problem.

In [ ]:
import importlib
import json
import os
import random
import re
import time

importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

N_TRAIN = 4000
N_EVAL = 1000
N_EVAL_BASE = 400
N_PROBE = 250
EPOCHS = 2

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    N_TRAIN, N_EVAL, N_EVAL_BASE, N_PROBE, EPOCHS = 24, 8, 8, 8, 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda" or SMOKE, "No GPU found. Set Accelerator to GPU T4 x2 and re-run."

DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"

print("device:", DEVICE, "|", GPU_NAME)
print("torch:", torch.__version__)

## Step 2 — Load the dataset

BANKING77 contains 13,083 real online-banking questions, each labelled with one
of 77 intents. We read it straight from the Hugging Face Hub; no account or
access token is needed.

We use four slices of the data:

- `train_df` — what the model learns from
- `eval_df` — held-out questions the model never sees during training
- `base_df` — the first 400 rows of `eval_df`, used for the slower base-model runs
- `probe_df` — the first 250 rows of `eval_df`, used for the bias tests

In [ ]:
def load_banking77():
    mirror = "https://huggingface.co/datasets/mteb/banking77/resolve/main/"
    try:
        train = pd.read_json(mirror + "train.jsonl", lines=True)
        test = pd.read_json(mirror + "test.jsonl", lines=True)
        train = train.rename(columns={"label_text": "intent"})[["text", "intent"]]
        test = test.rename(columns={"label_text": "intent"})[["text", "intent"]]
        print("loaded from the mteb/banking77 mirror")
    except Exception as error:
        print("mirror failed:", error)
        print("falling back to the datasets library")
        from datasets import load_dataset

        raw = load_dataset("PolyAI/banking77", trust_remote_code=True)
        names = raw["train"].features["label"].names
        frames = []
        for split in ("train", "test"):
            frame = raw[split].to_pandas()
            frame["intent"] = frame["label"].map(lambda i: names[i])
            frames.append(frame[["text", "intent"]])
        train, test = frames

    intents = sorted(set(train["intent"]) | set(test["intent"]))
    assert len(intents) == 77, f"expected 77 intents, found {len(intents)}"
    return train.reset_index(drop=True), test.reset_index(drop=True), intents


train_full, test_full, LABELS = load_banking77()
print(f"train rows: {len(train_full)}  test rows: {len(test_full)}  intents: {len(LABELS)}")

train_df = train_full.sample(n=min(N_TRAIN, len(train_full)), random_state=SEED).reset_index(drop=True)
eval_df = test_full.sample(n=min(N_EVAL, len(test_full)), random_state=SEED).reset_index(drop=True)
base_df = eval_df.head(N_EVAL_BASE).reset_index(drop=True)
probe_df = eval_df.head(N_PROBE).reset_index(drop=True)

print()
print(train_df.head(5).to_string(index=False))

counts = train_df["intent"].value_counts()
print()
print(f"examples per intent in the training sample: min {counts.min()}, max {counts.max()}")

## Step 3 — Write the prompts

The model is a text generator, so we ask it a question and read back the text it
writes. Two prompts are needed.

The **short prompt** just asks for the intent. The fine-tuned model uses this,
because training teaches it which 77 labels exist.

The **long prompt** pastes all 77 labels into the question. The base model needs
this, because nothing has ever told it what the valid answers are. Giving each
system the prompt that suits it is what makes the comparison fair.

In [ ]:
SHORT_PROMPT = (
    "You are an intent classifier for a retail bank's support chat.\n"
    "Reply with the intent label only, in snake_case. No explanation.\n\n"
    "Customer message: {q}\n"
    "Intent:"
)

LONG_PROMPT = (
    "You are an intent classifier for a retail bank's support chat.\n"
    "Choose exactly one intent from this list and reply with that label only, "
    "in snake_case. No explanation.\n\n"
    "Valid intents:\n" + "\n".join(LABELS) + "\n\n"
    "Customer message: {q}\n"
    "Intent:"
)

print(SHORT_PROMPT.format(q="my card still has not arrived"))

## Step 4 — Turn generated text into a label

The model can write anything at all, so we have to check whether what it wrote is
actually one of the 77 labels. `normalise` lowercases the text and turns spaces
and punctuation into underscores, so `"Card Arrival."` and `"card_arrival"` match.

If the text matches no label we return `None` and count it as wrong. A real
support system cannot route an answer it does not recognise, so treating a
near-miss as correct would flatter the model.

In [ ]:
NON_LETTERS = re.compile(r"[^a-z0-9]+")


def normalise(text):
    return NON_LETTERS.sub("_", text.strip().lower()).strip("_")


LABEL_LOOKUP = {normalise(label): label for label in LABELS}


def parse_prediction(generated_text):
    first_line = generated_text.strip().split("\n")[0].strip().strip('."\' ')
    key = normalise(first_line)

    if key in LABEL_LOOKUP:
        return LABEL_LOOKUP[key]

    for normalised_label, label in LABEL_LOOKUP.items():
        shorter, longer = sorted([key, normalised_label], key=len)
        if longer.startswith(shorter) and len(longer) - len(shorter) <= 6:
            return label

    return None


for sample in ["card_arrival", "Card Arrival.", "lost_or_stolen_card\nbecause it was stolen", "hello there"]:
    print(f"{sample!r:40} -> {parse_prediction(sample)}")

## Step 5 — Build the scoring functions

`predict` asks the model for an answer to every question. It works in batches,
because a GPU is far faster when it handles many questions at once.

Short and long questions are mixed together, and every question in a batch gets
padded to the length of the longest one. Sorting the questions by length first
means each batch holds questions of a similar size, so much less of the GPU's work
is wasted on padding. We sort, generate, then put the answers back in their
original order.

`score` reports four things:

- **accuracy** — how often the predicted label is the correct one
- **macro F1** — the average F1 across all 77 intents, so rare intents count as
  much as common ones
- **invalid label rate** — how often the model wrote something that is not a label
- **seconds per query** — how slow it is, which matters for a live chat

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_prompt(template, question):
    messages = [{"role": "user", "content": template.format(q=question)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def predict(model, questions, template, batch_size=32, max_new_tokens=16):
    model.eval()
    tokenizer.padding_side = "left"

    prompts = [build_prompt(template, question) for question in questions]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    answers = [None] * len(prompts)

    start = time.time()
    for batch_start in range(0, len(order), batch_size):
        positions = order[batch_start:batch_start + batch_size]
        batch = tokenizer(
            [prompts[i] for i in positions],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model.device)

        generated = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

        prompt_length = batch["input_ids"].shape[1]
        for row, position in enumerate(positions):
            new_tokens = generated[row][prompt_length:]
            answers[position] = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answers, time.time() - start


def macro_f1(true_labels, predicted_labels):
    scores = []
    for label in sorted(set(true_labels)):
        hits = sum(1 for t, p in zip(true_labels, predicted_labels) if t == label and p == label)
        false_alarms = sum(1 for t, p in zip(true_labels, predicted_labels) if t != label and p == label)
        misses = sum(1 for t, p in zip(true_labels, predicted_labels) if t == label and p != label)

        precision = hits / (hits + false_alarms) if hits + false_alarms else 0.0
        recall = hits / (hits + misses) if hits + misses else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        scores.append(f1)

    return float(np.mean(scores))


def score(name, model, data, template, batch_size=32):
    raw_answers, seconds = predict(model, data["text"].tolist(), template, batch_size)
    predictions = [parse_prediction(answer) for answer in raw_answers]
    true_labels = data["intent"].tolist()

    summary = {
        "system": name,
        "n": len(data),
        "accuracy": round(float(np.mean([p == t for p, t in zip(predictions, true_labels)])), 4),
        "macro_f1": round(macro_f1(true_labels, [p or "none" for p in predictions]), 4),
        "invalid_label_rate": round(float(np.mean([p is None for p in predictions])), 4),
        "sec_per_query": round(seconds / len(data), 3),
    }

    print(json.dumps(summary, indent=2))
    return summary, predictions, raw_answers


RESULTS = {
    "config": {
        "model": MODEL_ID,
        "n_train": N_TRAIN,
        "epochs": EPOCHS,
        "n_eval": N_EVAL,
        "n_eval_base": N_EVAL_BASE,
        "seed": SEED,
        "gpu": GPU_NAME,
    }
}

## Step 6 — Measure the model before training

This is the "before" half of the experiment. System A gets the short prompt, so we
find out whether the model has any idea what the 77 labels are. System B gets the
full list, which is what you would actually deploy if you had no training data.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)

RESULTS["A_base_short_prompt"], _, raw_answers_a = score(
    "base / short prompt", base_model, base_df, SHORT_PROMPT
)

print()
print("What the base model writes when it has not seen the label list:")
for answer in raw_answers_a[:5]:
    print("  ", repr(answer.strip()[:70]))

In [ ]:
RESULTS["B_base_label_list"], _, _ = score(
    "base / 77-label prompt", base_model, base_df, LONG_PROMPT, batch_size=16
)

del base_model
torch.cuda.empty_cache()

## Step 7 — Prepare the training examples

A language model learns by predicting the next token. For each training row we
build the full exchange — the question, then the correct label as the answer — and
ask the model to predict it.

One detail matters. We set the question's tokens to `-100`, which tells the
training loop to ignore them when measuring error. Without this the model would
spend most of its effort learning to write banking questions, when the only thing
we want it to learn is which label follows.

In [ ]:
class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, data, max_length=160):
        self.examples = []

        for question, intent in zip(data["text"], data["intent"]):
            prompt = build_prompt(SHORT_PROMPT, question)
            prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            answer_ids = tokenizer(" " + intent + "<|im_end|>", add_special_tokens=False)["input_ids"]

            self.examples.append({
                "input_ids": (prompt_ids + answer_ids)[:max_length],
                "labels": ([-100] * len(prompt_ids) + answer_ids)[:max_length],
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate(batch):
    longest = max(len(item["input_ids"]) for item in batch)
    pad_id = tokenizer.pad_token_id

    return {
        "input_ids": torch.tensor([
            item["input_ids"] + [pad_id] * (longest - len(item["input_ids"])) for item in batch
        ]),
        "attention_mask": torch.tensor([
            [1] * len(item["input_ids"]) + [0] * (longest - len(item["input_ids"])) for item in batch
        ]),
        "labels": torch.tensor([
            item["labels"] + [-100] * (longest - len(item["labels"])) for item in batch
        ]),
    }


train_dataset = IntentDataset(train_df)
lengths = [len(example["input_ids"]) for example in train_dataset.examples]
print("training examples:", len(train_dataset))
print("median tokens per example:", int(np.median(lengths)))

## Step 8 — Fine-tune with LoRA

Training all 494 million weights would need far more memory and time than a free
GPU gives us. LoRA avoids that: it freezes the pretrained weights and inserts a
small pair of trainable matrices beside each one. We end up training roughly 9
million weights, under 2% of the model, and everything else stays exactly as Qwen
shipped it.

The base model is loaded in 32-bit here while training runs in mixed precision,
because training LoRA on top of a 16-bit base tends to produce `nan` losses.

We train on 8 examples at a time and only update the weights after 4 of those
batches, which keeps memory low while still learning from 32 examples per update.

Watch the loss in the output below. It should fall steadily.

In [ ]:
import inspect

import transformers
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

print("transformers version:", transformers.__version__)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

requested_arguments = {
    "output_dir": f"{OUT_DIR}/checkpoints",
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": EPOCHS,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_strategy": "no",
    "fp16": (DEVICE == "cuda"),
    "report_to": "none",
    "seed": SEED,
}

signature = inspect.signature(TrainingArguments.__init__)
accepts_any_keyword = any(p.kind == p.VAR_KEYWORD for p in signature.parameters.values())

if accepts_any_keyword:
    supported_arguments = dict(requested_arguments)
else:
    supported_arguments = {
        name: value for name, value in requested_arguments.items() if name in signature.parameters
    }

skipped = sorted(set(requested_arguments) - set(supported_arguments))
if skipped:
    print("this transformers version does not accept:", skipped)

if "warmup_ratio" in skipped and "warmup_steps" in signature.parameters:
    supported_arguments["warmup_steps"] = 10

for essential in ["num_train_epochs", "learning_rate"]:
    assert essential in supported_arguments, f"{essential} is not supported, cannot train reliably"

training_arguments = TrainingArguments(**supported_arguments)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    data_collator=collate,
)

start_time = time.time()
training_output = trainer.train()
training_minutes = (time.time() - start_time) / 60

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

RESULTS["training"] = {
    "minutes": round(training_minutes, 2),
    "final_loss": round(float(training_output.training_loss), 4),
    "trainable_params": int(trainable),
    "total_params": int(total),
    "percent_trainable": round(100 * trainable / total, 2),
    "loss_curve": [
        {"step": entry["step"], "loss": round(entry["loss"], 4)}
        for entry in trainer.state.log_history
        if "loss" in entry
    ],
}

print(RESULTS["training"])
model.save_pretrained(f"{OUT_DIR}/banking77-adapter")

## Step 9 — Measure the model after training

`merge_and_unload` folds the trained LoRA matrices back into the model's weights,
giving us one ordinary model to test.

We score it twice. Once on the full 1,000-row test sample, which is the headline
result, and once on the same 400 rows the base model saw, so the before-and-after
comparison is on identical questions.

In [ ]:
model.config.use_cache = True
tuned_model = model.merge_and_unload()
tuned_model = (tuned_model.half() if DEVICE == "cuda" else tuned_model).eval()

del model
torch.cuda.empty_cache()

RESULTS["C_finetuned"], predictions_c, _ = score(
    "lora / short prompt", tuned_model, eval_df, SHORT_PROMPT
)

RESULTS["C_finetuned_same_subset"], _, _ = score(
    "lora / short prompt (base subset)", tuned_model, base_df, SHORT_PROMPT
)

## Step 10 — Find out which intents fail

An average hides the failures. Here we group the test results by intent to see
which ones the model gets wrong, and which pairs of intents it mixes up. These go
into the limitations section of the report.

In [ ]:
outcomes = pd.DataFrame({"true": eval_df["intent"], "predicted": predictions_c})
outcomes["correct"] = outcomes["true"] == outcomes["predicted"]

per_intent = (
    outcomes.groupby("true")
    .agg(n=("correct", "size"), accuracy=("correct", "mean"))
    .sort_values("accuracy")
)
per_intent.to_csv(f"{OUT_DIR}/per_intent_accuracy.csv")

RESULTS["worst_intents"] = [
    {"intent": intent, "n": int(row.n), "accuracy": round(float(row.accuracy), 3)}
    for intent, row in per_intent.head(8).iterrows()
]

print("Weakest intents:")
print(per_intent.head(8).to_string())

confusions = (
    outcomes[~outcomes["correct"]]
    .groupby(["true", "predicted"])
    .size()
    .sort_values(ascending=False)
    .head(8)
)

RESULTS["top_confusions"] = [
    {"true": true, "predicted_as": predicted, "count": int(count)}
    for (true, predicted), count in confusions.items()
]

print()
print("Most common mix-ups:")
print(confusions.to_string())

## Step 11 — Test for bias and brittleness

This is the fairness check. We take the same 250 test questions and rewrite them
in different styles. The customer's need has not changed, so the correct label has
not changed either. Any drop in accuracy means the model is reacting to *how*
somebody writes rather than *what* they need.

We test formal Indian English, Hindi-English code-mixing, typos, all caps, and
four different customer names. The name tests matter most: if attaching the name
Mohammed to a question changes the routing, that is discrimination, and the report
has to say so.

`flip_rate` counts how many predictions changed compared to the plain version.

In [ ]:
HINGLISH_WORDS = {
    "my": "mera",
    "money": "paisa",
    "the": "",
    "i": "main",
    "need": "chahiye",
    "please": "kripya",
    "not": "nahi",
    "how": "kaise",
    "why": "kyun",
    "when": "kab",
    "is": "hai",
    "are": "hain",
    "can": "sakta",
    "do": "karo",
    "get": "milega",
    "want": "chahiye",
    "working": "kaam nahi kar raha",
}


def to_hinglish(text):
    return " ".join(HINGLISH_WORDS.get(word.lower(), word) for word in text.split()).strip()


def to_indian_english(text):
    return f"Sir kindly do the needful, {text[0].lower() + text[1:]} only."


def add_typos(text, rate=0.06):
    generator = random.Random(SEED)
    characters = [character for character in text]
    for index, character in enumerate(characters):
        if character.isalpha() and generator.random() < rate:
            characters[index] = ""
    return "".join(characters)


def add_name(name):
    return lambda text: f"Hi, this is {name}. {text}"


PROBES = {
    "plain": lambda text: text,
    "indian_english": to_indian_english,
    "hinglish": to_hinglish,
    "typos": add_typos,
    "shouting_caps": lambda text: text.upper(),
    "name_arjun": add_name("Arjun"),
    "name_ananya": add_name("Ananya"),
    "name_mohammed": add_name("Mohammed"),
    "name_john": add_name("John"),
}

probe_results = {}
plain_predictions = None

for probe_name, rewrite in PROBES.items():
    rewritten = probe_df.assign(text=probe_df["text"].map(rewrite))
    raw_answers, _ = predict(tuned_model, rewritten["text"].tolist(), SHORT_PROMPT)
    predictions = [parse_prediction(answer) for answer in raw_answers]
    true_labels = rewritten["intent"].tolist()

    if plain_predictions is None:
        plain_predictions = predictions

    probe_results[probe_name] = {
        "accuracy": round(float(np.mean([p == t for p, t in zip(predictions, true_labels)])), 4),
        "invalid_label_rate": round(float(np.mean([p is None for p in predictions])), 4),
        "flip_rate": round(float(np.mean([a != b for a, b in zip(predictions, plain_predictions)])), 4),
    }

    print(f"{probe_name:16} {probe_results[probe_name]}")

plain_accuracy = probe_results["plain"]["accuracy"]
for probe in probe_results.values():
    probe["change_in_points"] = round((probe["accuracy"] - plain_accuracy) * 100, 2)

RESULTS["probes"] = probe_results

name_accuracies = {
    name: probe["accuracy"] for name, probe in probe_results.items() if name.startswith("name_")
}
RESULTS["name_swap_spread_points"] = round(
    (max(name_accuracies.values()) - min(name_accuracies.values())) * 100, 2
)

print()
print("Accuracy gap between customer names, in points:", RESULTS["name_swap_spread_points"])

## Step 12 — Save everything

Download `results.json` and `per_intent_accuracy.csv` from the Kaggle output
panel on the right. The report is written from these two files.

In [ ]:
with open(f"{OUT_DIR}/results.json", "w") as results_file:
    json.dump(RESULTS, results_file, indent=2)

headline = pd.DataFrame([
    RESULTS["A_base_short_prompt"],
    RESULTS["B_base_label_list"],
    RESULTS["C_finetuned_same_subset"],
    RESULTS["C_finetuned"],
])

print("HEADLINE RESULTS")
print(headline.to_string(index=False))

print()
print("BIAS AND ROBUSTNESS")
print(pd.DataFrame(probe_results).T.to_string())

print()
print(f"training took {RESULTS['training']['minutes']} minutes")
print(f"trained {RESULTS['training']['trainable_params']:,} weights "
      f"({RESULTS['training']['percent_trainable']}% of the model)")
print()
print("saved:", f"{OUT_DIR}/results.json", "and", f"{OUT_DIR}/per_intent_accuracy.csv")

## Step 13 — Try it yourself

These questions are not in the dataset. This is the cell to run during a demo.

In [ ]:
demo_questions = [
    "I tapped my card at the metro gate and got charged twice",
    "why is there a fee for sending money to my friend in canada",
    "my salary hasn't shown up yet and rent is due tomorrow",
    "someone used my card in another city, I still have it with me",
    "can I hold euros and rupees in the same account",
]

demo_answers, _ = predict(tuned_model, demo_questions, SHORT_PROMPT)

for question, answer in zip(demo_questions, demo_answers):
    print(question)
    print("   ->", parse_prediction(answer))
    print()